# Tutorial 2: Correct a full spectrum

This tutorial corrects a reduced, stitched HARPS spectrum
containing 313,090 samples.

It first runs a minimal automatic correction. Fit and exclusion
ranges are introduced only after that baseline has been inspected.
PyMolFit automatically divides the radiative-transfer calculation
into manageable wavelength segments while returning one spectrum
on the original sampling.

## Install

Install PyMolFit and interactive plotting support in the
environment used by this notebook:

```bash
python -m pip install pymolfit ipympl
```

In [ ]:
%matplotlib widget

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pymolfit import correct, load_spectrum

In [ ]:
candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the PyMolFit repository or tutorials directory"
    )

INPUT = TUTORIAL_ROOT / "data" / "ADP.2017-04-07T01_04_41.632.fits"
spectrum = load_spectrum(
    INPUT,
    wavelength_medium="air",
).to_air().to_unit("angstrom")

print(f"Pixels: {spectrum.wavelength.size:,}")
print(
    f"Wavelength range: {spectrum.wavelength[0]:.1f}-"
    f"{spectrum.wavelength[-1]:.1f} Angstrom"
)

## Inspect the complete original spectrum

Every valid input pixel is plotted. Interactive zooming is useful
for inspecting individual orders and absorption bands.

In [ ]:
valid = spectrum.valid & np.isfinite(spectrum.flux)
scale = np.nanmedian(spectrum.flux[valid])

plt.figure(figsize=(13, 4))
plt.plot(
    spectrum.wavelength[valid],
    spectrum.flux[valid] / scale,
    color="black",
    linewidth=0.35,
)
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Complete original HARPS spectrum")
plt.tight_layout()
plt.show()

## First correction: automatic baseline

This call supplies only the input and wavelength medium. Automatic
segmentation, atmosphere selection, molecular line selection,
continuum fitting, instrumental broadening, and wavelength
alignment remain at their defaults.

In [ ]:
baseline_result = correct(
    input_path=INPUT,
    wavelength_medium="air",
)

if not baseline_result.success:
    raise RuntimeError(baseline_result.message)

## Inspect the baseline result

This full-resolution view shows what the automatic fit does before
any spectral regions are selected manually.

In [ ]:
baseline_observed = baseline_result.spectrum.to_air().to_unit("angstrom")
baseline_corrected = baseline_result.corrected.to_air().to_unit("angstrom")
scale = np.nanmedian(
    baseline_observed.flux[
        baseline_observed.valid & np.isfinite(baseline_observed.flux)
    ]
)

figure, axes = plt.subplots(
    2,
    1,
    figsize=(13, 7),
    sharex=True,
    height_ratios=(2, 1),
    constrained_layout=True,
)
axes[0].plot(
    baseline_observed.wavelength,
    baseline_observed.flux / scale,
    color="black",
    linewidth=0.35,
    alpha=0.55,
    label="Observed",
)
axes[0].plot(
    baseline_corrected.wavelength,
    baseline_corrected.flux / scale,
    color="tab:blue",
    linewidth=0.35,
    label="Baseline corrected",
)
axes[0].set_ylabel("Flux / median")
axes[0].legend()

axes[1].plot(
    baseline_observed.wavelength,
    baseline_result.transmission,
    color="tab:red",
    linewidth=0.35,
)
axes[1].set_xlabel("Air wavelength [Angstrom]")
axes[1].set_ylabel("Transmission")
axes[1].set_ylim(-0.02, 1.05)

figure.suptitle("Full-spectrum automatic baseline")
plt.show()

## Choose fit and exclusion regions

A broad stellar spectrum should not use every pixel to estimate
atmospheric and instrumental parameters. Fit ranges select useful
telluric information. The exclusion range protects the
astrophysical Na D absorption.

These values are unshifted air wavelengths in microns, matching
the input spectrum. They are displayed on the original spectrum
before the next correction is run.

In [ ]:
FIT_RANGES = (
    (0.5878, 0.5982),  # H2O-rich optical interval containing Na D
    (0.6275, 0.6285),  # Additional H2O information
    (0.6866, 0.6880),  # O2 B band
)

EXCLUDE_RANGES = (
    (0.58875, 0.58996),  # Broad astrophysical Na D absorption
)

figure, axes = plt.subplots(
    2,
    1,
    figsize=(13, 8),
    constrained_layout=True,
)
axes[0].plot(
    spectrum.wavelength[valid],
    spectrum.flux[valid] / scale,
    color="black",
    linewidth=0.35,
)
for index, (lower, upper) in enumerate(FIT_RANGES):
    axes[0].axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:green",
        alpha=0.12,
        label="Fit range" if index == 0 else None,
    )
for index, (lower, upper) in enumerate(EXCLUDE_RANGES):
    axes[0].axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:red",
        alpha=0.16,
        label="Excluded from fit" if index == 0 else None,
    )
axes[0].set_xlabel("Air wavelength [Angstrom]")
axes[0].set_ylabel("Flux / median")
axes[0].set_title("Selected regions on the complete original spectrum")
axes[0].legend()

for index, (lower, upper) in enumerate(FIT_RANGES):
    selection = (
        valid
        & (spectrum.wavelength >= lower * 1e4)
        & (spectrum.wavelength <= upper * 1e4)
    )
    axes[1].plot(
        spectrum.wavelength[selection],
        spectrum.flux[selection] / scale,
        linewidth=0.55,
        label=f"{lower * 1e4:.0f}-{upper * 1e4:.0f} Angstrom",
    )
for lower, upper in EXCLUDE_RANGES:
    axes[1].axvspan(
        lower * 1e4,
        upper * 1e4,
        color="tab:red",
        alpha=0.16,
    )
axes[1].set_xlabel("Air wavelength [Angstrom]")
axes[1].set_ylabel("Flux / median")
axes[1].set_title("Original pixels available inside the fit ranges")
axes[1].legend()

plt.show()

## Second correction: use the inspected regions

Only the fit and exclusion masks are added to the minimal call.
PyMolFit continues to calculate transmission over the complete
spectrum.

In [ ]:
refined_result = correct(
    input_path=INPUT,
    wavelength_medium="air",
    fit_ranges=FIT_RANGES,
    exclude_ranges=EXCLUDE_RANGES,
)

if not refined_result.success:
    raise RuntimeError(refined_result.message)

## Compare the complete results

All 313,090 samples are plotted. The corrected spectrum has the
same wavelength grid as the original.

In [ ]:
refined_corrected = refined_result.corrected.to_air().to_unit("angstrom")

plt.figure(figsize=(13, 5))
plt.plot(
    baseline_observed.wavelength,
    baseline_observed.flux / scale,
    color="black",
    linewidth=0.3,
    alpha=0.45,
    label="Observed",
)
plt.plot(
    baseline_corrected.wavelength,
    baseline_corrected.flux / scale,
    color="tab:orange",
    linewidth=0.3,
    label="Automatic baseline",
)
plt.plot(
    refined_corrected.wavelength,
    refined_corrected.flux / scale,
    color="tab:blue",
    linewidth=0.3,
    label="Using fit masks",
)
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux / median")
plt.title("Complete before-and-after comparison")
plt.legend()
plt.tight_layout()
plt.show()

## Inspect each fitted interval

These full-resolution panels show the original spectrum, both
corrections, and the refined atmospheric transmission. Very deep
line cores may remain unreliable because little astronomical
signal survives there.

In [ ]:
figure, axes = plt.subplots(
    len(FIT_RANGES),
    1,
    figsize=(13, 10),
    constrained_layout=True,
)

for axis, (lower, upper) in zip(axes, FIT_RANGES, strict=True):
    lower_angstrom = lower * 1e4
    upper_angstrom = upper * 1e4
    selection = (
        (baseline_observed.wavelength >= lower_angstrom)
        & (baseline_observed.wavelength <= upper_angstrom)
    )

    axis.plot(
        baseline_observed.wavelength[selection],
        baseline_observed.flux[selection] / scale,
        color="black",
        linewidth=0.55,
        alpha=0.55,
        label="Observed",
    )
    axis.plot(
        baseline_corrected.wavelength[selection],
        baseline_corrected.flux[selection] / scale,
        color="tab:orange",
        linewidth=0.65,
        label="Automatic baseline",
    )
    axis.plot(
        refined_corrected.wavelength[selection],
        refined_corrected.flux[selection] / scale,
        color="tab:blue",
        linewidth=0.65,
        label="Using fit masks",
    )
    axis.plot(
        baseline_observed.wavelength[selection],
        refined_result.transmission[selection],
        color="tab:red",
        linestyle="--",
        linewidth=0.65,
        label="Refined transmission",
    )
    for exclude_lower, exclude_upper in EXCLUDE_RANGES:
        axis.axvspan(
            exclude_lower * 1e4,
            exclude_upper * 1e4,
            color="tab:red",
            alpha=0.10,
        )
    axis.set_xlim(lower_angstrom, upper_angstrom)
    axis.set_ylabel("Relative flux")

axes[0].legend(ncol=2)
axes[-1].set_xlabel("Air wavelength [Angstrom]")
figure.suptitle("Full-resolution fit-region diagnostics")
plt.show()

## Saving is optional

Correction and saving are separate operations. Save only the
corrected spectrum with
`save_corrected_txt(refined_result, "corrected.txt")`, or retain
the complete auditable result with
`save_fit_product_ecsv(refined_result, "fit_product.ecsv")`.